## Действия с тензорами

Тензор - базовая структура данных в PyTorch. Его можно воспринимать как обобщение обычного числа, списка чисел и таблицы чисел. В машинном обучении почти все данные, параметры модели и промежуточные результаты представлены именно тензорами.

В этом разделе удобно держать в голове три характеристики тензора: `shape`, `dtype` и сами значения. `shape` говорит, как данные разложены по осям, `dtype` - как именно числа хранятся в памяти, а значения - это конкретное содержимое. Многие ошибки в нейросетях возникают не из-за формулы, а из-за несовпадения форм тензоров.


In [1]:
import torch
torch.__version__

'2.12.0'

Создаём тензоры разных размерностей: 0D, 1D и 2D.

В терминах размерности:

- `tensor0d` - скаляр, то есть одно число без осей;
- `tensor1d` - вектор, то есть последовательность чисел вдоль одной оси;
- `tensor2d` - матрица, то есть таблица со строками и столбцами.

Такая иерархия потом масштабируется дальше: изображения часто хранятся как 3D или 4D тензоры, а батчи текстов и эмбеддингов - как тензоры еще большей размерности.


In [2]:
tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2, 3])
tensor2d = torch.tensor([[1, 2, 3], [4, 5, 6]])

Выводим содержимое двумерного тензора.

При выводе `tensor2d` PyTorch показывает не только значения, но и структуру: две строки и три столбца. Это помогает читать тензор как маленькую таблицу, где каждая строка может быть отдельным объектом, а каждый столбец - отдельным признаком.


In [3]:
tensor2d

tensor([[1, 2, 3],
        [4, 5, 6]])

Смотрим тип данных (dtype) элементов тензора.
`dtype` определяет, какие числа может хранить тензор и как они будут участвовать в вычислениях. Например, целочисленные тензоры подходят для индексов и меток классов, а вещественные типы вроде `float32` обычно нужны для весов, входных признаков и градиентов.


In [4]:
tensor2d.dtype

torch.int64

Преобразуем тензор к типу float32 и выводим результат.

Преобразование к `float32` особенно важно для нейронных сетей. Большинство параметров модели и операций обучения работают с числами с плавающей точкой, потому что градиенты почти всегда дробные. Если оставить данные целыми числами, часть операций обучения просто не сможет корректно считать производные.


In [5]:
floatvec = tensor2d.to(torch.float32)
floatvec

tensor([[1., 2., 3.],
        [4., 5., 6.]])

Получаем форму (shape) двумерного тензора.

Для `tensor2d` форма читается как `(количество строк, количество столбцов)`. В данном примере это `2 x 3`: две строки и три значения в каждой строке. В задачах машинного обучения первая ось часто означает количество объектов в батче, а последняя - количество признаков у каждого объекта.


In [6]:
tensor2d.shape

torch.Size([2, 3])

Меняем форму тензора с помощью view без изменения данных.
`view(3, 2)` меняет только способ интерпретации тех же самых шести чисел. Было `2 x 3`, стало `3 x 2`, но общее количество элементов осталось тем же: `2 * 3 = 3 * 2 = 6`. Это важное правило для reshape-операций: новая форма должна вмещать ровно столько же элементов.


In [7]:
tensor2d.view(3, 2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

Транспонируем двумерный тензор, меняя местами строки и столбцы.

Транспонирование превращает строки в столбцы, а столбцы - в строки. Если исходная форма была `2 x 3`, после `.T` она станет `3 x 2`. Эта операция особенно часто нужна перед матричным умножением, потому что внутренние размерности матриц должны совпадать.


In [8]:
tensor2d.T

tensor([[1, 4],
        [2, 5],
        [3, 6]])

Разные способы умножения матриц с т.з. синтаксиса

Обе записи ниже выполняют матричное умножение. Разница только в стиле: `.matmul(...)` выглядит как явный вызов метода, а оператор `@` короче и обычно читается ближе к математической записи.

В примере умножается тензор формы `2 x 3` на транспонированный тензор формы `3 x 2`. Внутренние размерности `3` и `3` совпадают, поэтому операция допустима, а результат получает форму `2 x 2`.


In [9]:
tensor2d.matmul(tensor2d.T)

tensor([[14, 32],
        [32, 77]])

Здесь та же самая операция записана через оператор `@`. В PyTorch это привычный синтаксис для матричного умножения, и в учебном коде он часто помогает быстрее увидеть саму математическую идею.

```text
[2 x 3] @ [3 x 2] -> [2 x 2]
```

Средние размерности должны совпасть, а внешние размерности становятся формой результата.


In [10]:
tensor2d @ tensor2d.T

tensor([[14, 32],
        [32, 77]])

## Логистрическая регрессия

В этом разделе тензорные операции связываются с первой маленькой моделью. Логистическая регрессия хороша как учебный мост: в ней уже есть параметры, функция активации, функция потерь и обратное распространение ошибки, но формула еще остается достаточно простой для ручного чтения.


### Как работает логистическая регрессия

В этом примере показана самая простая логистическая регрессия с одним входным признаком. Такая модель нужна для бинарной классификации, когда мы хотим получить ответ в духе `0` или `1`, `нет` или `да`.

**Что означают переменные:**
- `x1` - входной признак, по которому модель делает предсказание;
- `w1` - вес, который показывает силу влияния признака на результат;
- `b` - смещение (`bias`), которое помогает двигать границу решения;
- `y` - правильная целевая метка, с которой сравнивается ответ модели.

**Как идет вычисление:**
- сначала считается линейная часть: `z = x1 * w1 + b`;
- затем к `z` применяется `sigmoid`, и получается `a` - число от `0` до `1`;
- это значение можно понимать как вероятность положительного класса;
- после этого `binary_cross_entropy(a, y)` считает ошибку между предсказанием и правильным ответом.

**Как работает `sigmoid`:**

`sigmoid(z) = 1 / (1 + e^(-z))`

Эта функция переводит любое число в диапазон от `0` до `1`. Если `z` большое и положительное, результат будет ближе к `1`. Если `z` отрицательное, результат будет ближе к `0`. Поэтому `sigmoid` удобно использовать там, где модель должна выдать вероятность.

**Как работает `binary_cross_entropy`:**

Эта функция потерь сравнивает вероятность `a` с правильной меткой `y`. Если модель предсказала верно и уверенно, ошибка будет маленькой. Если модель ошиблась, значение функции потерь вырастет. Обучение как раз и нужно для того, чтобы уменьшать эту ошибку.

**Зачем здесь градиенты:**
- `w1` и `b` созданы с `requires_grad=True`, поэтому PyTorch отслеживает все вычисления с ними;
- после вызова `loss.backward()` библиотека автоматически считает производные ошибки по этим параметрам;
- найденные градиенты попадают в `w1.grad` и `b.grad`.

**Схема вычислений:**

```text
x1
  -> z = x1 * w1 + b
  -> sigmoid(z)
  -> a = predicted probability
  -> binary_cross_entropy(a, y)
  -> loss
```

То есть логистическая регрессия сначала считает линейную комбинацию признака, потом превращает ее в вероятность и сравнивает эту вероятность с правильным ответом.
### 

### Почему это уже обучение

До вызова `backward()` мы просто выполняем прямые вычисления: из `x1`, `w1` и `b` получаем `z`, потом `a`, потом `loss`. Но поскольку `w1` и `b` требуют градиенты, PyTorch параллельно строит граф вычислений. В этом графе каждая операция знает, как передать вклад ошибки назад к своим входам.

Когда вызывается `loss.backward()`, происходит обратный проход по этому графу. PyTorch применяет правило цепочки и отвечает на вопрос: насколько изменится ошибка, если немного изменить `w1` или `b`. Именно эти величины потом используются оптимизатором, чтобы сдвинуть параметры в сторону меньшей ошибки.

Важно помнить: `.backward()` не меняет веса сам по себе. Он только заполняет поля `.grad`. Шаг обновления параметров обычно делает оптимизатор, например `torch.optim.SGD` или `torch.optim.Adam`.


In [11]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0]) # целевая метка
x1 = torch.tensor([1.1]) # входной признак
w1 = torch.tensor([2.2], requires_grad=True) # весовой параметр
b = torch.tensor([0.0], requires_grad=True) # единица смещения bias

z = x1 * w1 + b # вход сети
a = torch.sigmoid(z) # функция активация примененная ко входу (выход)

loss = F.binary_cross_entropy(a, y) # активация и выход

# grad_L_w1 = grad(loss, w1, retain_graph=True)
# grad_L_b = grad(loss, b, retain_graph=True)

# print(grad_L_w1)
# print(grad_L_b)

Теперь запускается обратное распространение ошибки. После `loss.backward()` у параметров, которые участвовали в вычислении `loss` и имели `requires_grad=True`, появляются значения в `.grad`.

Здесь выводятся два градиента:

- `w1.grad` - насколько ошибка чувствительна к весу признака;
- `b.grad` - насколько ошибка чувствительна к смещению.

В реальном цикле обучения перед новым `backward()` градиенты обычно обнуляют, потому что PyTorch по умолчанию накапливает их, а не заменяет.


In [12]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


## Реализация многослойных нейронных сетей

Теперь вместо одной формулы строится сеть из нескольких слоев. Каждый слой выполняет свое преобразование, а вся модель собирается как последовательность таких преобразований. Это уже ближе к тому, как обычно пишут модели на PyTorch: мы описываем архитектуру один раз, а потом вызываем модель как функцию.


### Как работает класс `NewralNetwork`

Этот класс наследуется от `torch.nn.Module`, поэтому PyTorch воспринимает его как полноценную нейронную сеть. Внутри класса есть две главные части: `__init__` описывает, из каких слоев состоит сеть, а `forward` задает путь, по которому входные данные проходят через эти слои.

**Что происходит в `self.layers`:**
- `Linear(num_inputs, 30)` преобразует входной вектор в 30 признаков первого скрытого слоя.
- `ReLU()` добавляет нелинейность, чтобы сеть могла учить более сложные зависимости, а не только линейные преобразования.
- `Linear(30, 20)` строит второй скрытый слой из 20 нейронов.
- `ReLU()` снова оставляет положительные значения и обнуляет отрицательные.
- `Linear(20, num_outputs)` формирует итоговый выход модели.

**Как работает `ReLU`:**

`ReLU(x) = max(0, x)`

Если на вход приходит отрицательное число, функция возвращает `0`. Если число положительное, оно проходит дальше без изменений. Благодаря этому сеть становится нелинейной: без `ReLU` несколько слоев `Linear` подряд свелись бы почти к одному линейному преобразованию.

**Как работает `forward`:**
- когда мы пишем `model(x)`, PyTorch автоматически вызывает `forward(x)`;
- `x` проходит через всю цепочку слоев, записанную в `self.layers`;
- на выходе получается `logits` - сырое предсказание модели до применения `softmax` или другой функции активации на выходе;
- затем `forward` возвращает это значение наружу.

Именно поэтому блок

```python
def forward(self, x):
    logits = self.layers(x)
    return logits
```

означает: взять входной тензор `x`, пропустить его через все слои сети и вернуть результат.

**Схема архитектуры сети:**

```text
x (num_inputs)
    -> Linear(num_inputs, 30)
    -> ReLU
    -> Linear(30, 20)
    -> ReLU
    -> Linear(20, num_outputs)
    -> logits
```

То есть сеть берет входные признаки, постепенно преобразует их в более удобное внутреннее представление и в конце выдает итоговые значения для предсказания.
### 

### Как читать размерности в этой архитектуре

Если создать `NeuralNetwork(50, 3)`, то сеть ожидает на вход 50 признаков и возвращает 3 выходных значения. Внутри получается такая цепочка форм:

```text
(1, 50) -> Linear(50, 30) -> (1, 30)
        -> ReLU           -> (1, 30)
        -> Linear(30, 20) -> (1, 20)
        -> ReLU           -> (1, 20)
        -> Linear(20, 3)  -> (1, 3)
```

`ReLU` не меняет форму тензора, она меняет только значения: отрицательные заменяет на нули, положительные оставляет. А вот `Linear` меняет последнюю размерность: было 50 признаков, стало 30; потом 30 превращаются в 20; в конце 20 превращаются в 3 выходных числа.

Каждый `Linear(in_features, out_features)` хранит два набора параметров: матрицу весов формы `[out_features, in_features]` и вектор смещений формы `[out_features]`. Поэтому число нейронов слоя видно по количеству строк в матрице весов.


In [13]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs): # входы и выходы
        super().__init__()
        self.layers = torch.nn.Sequential(
            # первый скрытый слой
            torch.nn.Linear(num_inputs, 30),
            # фукция активации между скрытыми слоями
            torch.nn.ReLU(),
            # кол-во входных словев = кол-во выходных слоев предыдущего слоя
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            # слой выходных данных
            torch.nn.Linear(20, num_outputs),
        )
    def forward(self, x):
        logits = self.layers(x)
        return logits

Создаем конкретный экземпляр сети: `NeuralNetwork(50, 3)`. Число `50` означает, что один объект описывается 50 входными признаками. Число `3` означает, что модель вернет 3 выходных значения.

Когда Jupyter выводит `model`, он показывает структуру модулей внутри сети. Это удобный способ проверить, что слои действительно собраны в нужном порядке и размерности совпадают.


In [14]:
model = NeuralNetwork(50, 3)
model

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

Эта строка считает общее число обучаемых параметров модели. `p.numel()` считает, сколько чисел хранится в каждом весе или `bias`, а `if p.requires_grad` оставляет только параметры, которые будут обучаться.

Для этой модели число параметров можно посчитать вручную:

```text
Linear(50, 30): 50 * 30 весов + 30 bias = 1530
Linear(30, 20): 30 * 20 весов + 20 bias = 620
Linear(20, 3):  20 * 3  весов + 3  bias = 63
Итого: 1530 + 620 + 63 = 2213
```

Это полезная проверка: если модель неожиданно содержит слишком много параметров, обучение может стать тяжелее и выше риск переобучения. Если параметров слишком мало, модели может не хватить гибкости, чтобы выучить зависимость в данных.


In [15]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of training parameters", num_params)

Total number of training parameters 2213


Смотрим веса первого линейного слоя. Это матрица, где каждая строка соответствует одному нейрону первого скрытого слоя, а каждый столбец - одному входному признаку.

Для `Linear(50, 30)` матрица весов имеет форму `[30, 50]`: 30 нейронов, и у каждого нейрона по 50 весов. Один нейрон берет все 50 входных чисел, умножает каждое на свой вес, складывает результаты и добавляет свой `bias`.


In [16]:
model.layers[0].weight

Parameter containing:
tensor([[ 0.1058, -0.0165, -0.0780,  ...,  0.0459, -0.0680, -0.0161],
        [-0.0543, -0.0879,  0.0943,  ...,  0.0121, -0.0440,  0.0746],
        [-0.1066,  0.0485, -0.1045,  ...,  0.1092,  0.1297,  0.0675],
        ...,
        [-0.0856,  0.1189,  0.1374,  ..., -0.1047,  0.1337,  0.0228],
        [ 0.1165, -0.0252, -0.0550,  ..., -0.1115, -0.1162,  0.0871],
        [-0.0403, -0.0053, -0.0169,  ..., -0.0655, -0.0950,  0.1035]],
       requires_grad=True)

Здесь отдельно выводится форма матрицы весов первого слоя. Это быстрый способ проверить правило PyTorch: у `Linear(in_features, out_features)` веса хранятся как `[out_features, in_features]`, а не наоборот.

```text
Linear(50, 30) -> weight.shape == [30, 50]
```


In [17]:
model.layers[0].weight.shape

torch.Size([30, 50])

Теперь смотрим `bias` первого слоя. У каждого из 30 нейронов есть свое отдельное смещение, поэтому вектор `bias` содержит 30 чисел.

Интуитивно `bias` позволяет нейрону сдвигать результат вверх или вниз независимо от входных признаков. Без смещения слой был бы менее гибким: все решения проходили бы через фиксированную точку относительно входов.


In [18]:
model.layers[0].bias

Parameter containing:
tensor([-0.0263,  0.0131,  0.0633,  0.0003, -0.0807, -0.0526,  0.1375,  0.0983,
         0.0466, -0.0063,  0.0259, -0.0041,  0.0932, -0.1201, -0.1252, -0.0510,
         0.0190, -0.0791, -0.0377, -0.0956, -0.1151, -0.0630, -0.0451,  0.1140,
         0.0082,  0.1208, -0.0866,  0.0087, -0.0307, -0.0874],
       requires_grad=True)

`torch.manual_seed(123)` фиксирует генератор случайных чисел. Это нужно, чтобы случайная инициализация весов повторялась при новом запуске ноутбука.

Для обучения это не делает модель “лучше”, но делает эксперимент воспроизводимым. Если результат неожиданно изменился, проще понять, дело в коде или просто в другой случайной инициализации.


In [19]:
torch.manual_seed(123) # делаем начальные веса воспроизводимыми
model = NeuralNetwork(50, 3)
model.layers[0].weight

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)

Создаем один входной пример `x` с формой `(1, 50)`. Первая размерность `1` - это размер батча: в батче сейчас один объект. Вторая размерность `50` - количество признаков у этого объекта.

Важно: даже если объект один, PyTorch-модель обычно получает данные с batch-осью. Поэтому используется форма `(1, 50)`, а не просто `(50,)`. Так сеть видит “один объект, у которого 50 признаков”.


In [20]:
x = torch.rand((1, 50))
x.shape

torch.Size([1, 50])

Здесь выводятся сами значения входного тензора `x`. Они случайные, потому что созданы через `torch.rand`, и лежат в диапазоне от `0` до `1`.

Для текущего примера смысл значений не важен: мы не обучаем модель на настоящих данных, а проверяем, как вход правильной формы проходит через сеть. В реальной задаче эти 50 чисел были бы признаками объекта, например измерениями, счетчиками или компонентами эмбеддинга.


In [21]:
x

tensor([[0.2391, 0.3194, 0.8111, 0.7507, 0.3306, 0.5374, 0.2845, 0.8459, 0.2232,
         0.2083, 0.8169, 0.1084, 0.3285, 0.7185, 0.3624, 0.3084, 0.8893, 0.4179,
         0.9741, 0.3697, 0.2397, 0.8936, 0.1443, 0.1365, 0.7625, 0.1632, 0.6641,
         0.1525, 0.9830, 0.5936, 0.9120, 0.0146, 0.6323, 0.4743, 0.7467, 0.3545,
         0.9994, 0.9815, 0.7399, 0.2057, 0.8742, 0.0138, 0.7676, 0.7481, 0.7570,
         0.6432, 0.9111, 0.2246, 0.8668, 0.6961]])

Прямой проход - это вычисление выходного тензора из входного. Запись `grad_fn=<AddmmBackward0>` означает, что этот тензор получился не просто как набор чисел, а как результат операции, для которой PyTorch запомнил правило обратного прохода. `Addmm` - по имени это исторически называется addmm, потому что это “add matrix-matrix multiply”, а не порядок чтения слева направо или справа налево: по смыслу это `x @ W.T + b`, то есть сначала матричное умножение, потом прибавление смещения.

Поэтому у `out` есть ссылка на узел графа вычислений. Когда позже вызовется `loss.backward()`, PyTorch пройдет по этому графу назад и посчитает градиенты для весов, bias и, если нужно, для входа. Если тензор не участвует в вычислениях с отслеживанием градиентов, у него будет `grad_fn=None`.

Для текущего входа форма результата будет `(1, 3)`: один объект в батче и три выходных числа для этого объекта. Эти числа называются `logits`, потому что они еще не превращены в вероятности. Для многоклассовой классификации после них часто применяют `softmax`, а для обучения используют функцию потерь, которая умеет работать с логитами напрямую.

Путь формы через сеть можно читать так:

```text
x:        (1, 50)
Linear 1: (1, 30)
ReLU:     (1, 30)
Linear 2: (1, 20)
ReLU:     (1, 20)
Linear 3: (1, 3)
out:      (1, 3)
```

Такой разбор помогает отлаживать почти любую нейросеть: если где-то форма изменилась не так, следующая матричная операция обычно сразу выдаст ошибку о несовместимых размерностях.


In [22]:
out = model(x)
out

tensor([[-0.1670,  0.1001, -0.1219]], grad_fn=<AddmmBackward0>)

Если мы используем модель для предсказания, а не для обучения, то не нужно отслеживать градиенты. Это делается с помощью контекстного менеджера `torch.no_grad()`

In [23]:
with torch.no_grad():
    out = model(x)
out

tensor([[-0.1670,  0.1001, -0.1219]])

В pyTorch принято программировать модели так, чтобы они возвращали выходные данные последнего слоя (логиты) до применения к ним функции активации

In [24]:
with torch.no_grad():
    out = torch.softmax(model(x), dim=1)
out # эти величины уже можно интерпретировать как вероятности принадлежности к классу

tensor([[0.2983, 0.3896, 0.3121]])

## Настройка эффективных загрузчиков данных

 pyTorch реализует классы Dataset - используется для создания обучающей и тестовой выборок, и Dataloader - определяет способ перетасовки данных и их объединения в пакеты 

In [27]:
# Создание набора данных
X_train = torch.tensor([
    [-1,2, 3,1],
    [-0,9, 2,9],
    [-0,5, 2,6],
    [2,3, -1,1],
    [2,7, -1,5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0,8, 2,8],
    [2,6, -1,6]
])
y_test = torch.tensor([0,1])

In [29]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, x, y):
        self.features = x
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [30]:
len(train_ds)

5